# Contextual Hierarchical Huber Screen

Stage 4 of the hierarchical modeling track. This notebook tests whether test-safe well descriptors can modulate row-level residual shape through regularized interactions, approximating a contextual hierarchy before any full Bayesian HSGP.

## 1. Setup

In [1]:
"Configure imports, paths, and contextual hierarchy priors."

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, RobustScaler, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

DATE_STAMP = "2026-06-23"
RUN_ID = f"{DATE_STAMP}__contextual_hierarchical_huber_screen"

ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for path in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS]:
    path.mkdir(parents=True, exist_ok=True)

RESIDUAL_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv"
WELL_AUDIT_PATH = ARTIFACT_TABLES / "model__hierarchical_residual_bias_well_summary__2026-06-22.csv"
TARGET_CANDIDATE = "huber_plus_hsgp_dps_m8"
LOCAL_REFERENCE_RMSE = 14.3586
PUBLIC_REFERENCE_RMSE = 14.514
OFFSET_CLIP = 12.0

ROW_FEATURES = ["eval_progress", "distance_log1p", "abs_projected_delta_10"]
CONTEXT_FEATURES = [
    "max_distance_log1p",
    "mean_distance_log1p",
    "xy_span",
    "known_tvt_fraction",
    "gr_missing_rate",
    "recent_tvt_slope_last_50",
    "mean_abs_projected_delta_10",
]

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-23__contextual_hierarchical_huber_screen


## 2. Load Residuals And Context

The target remains the out-of-fold residual from the current Huber + additive Hilbert model. Contextual hierarchy is represented by interactions between row coordinates and test-safe well descriptors.

In [2]:
"Load residual sample, merge well descriptors, and define baseline residuals."

residuals_all = pd.read_csv(RESIDUAL_PATH)
residuals = residuals_all.loc[residuals_all["candidate"].eq(TARGET_CANDIDATE)].copy()
for col in ["fold", "row_idx", "y_true", "y_pred", "residual", "eval_progress", "distance_from_boundary_md", "abs_projected_delta_10"]:
    residuals[col] = pd.to_numeric(residuals[col], errors="coerce")
residuals["distance_log1p"] = np.log1p(residuals["distance_from_boundary_md"].clip(lower=0))
residuals["baseline_pred"] = residuals["y_pred"]
residuals["baseline_residual"] = residuals["y_true"] - residuals["baseline_pred"]

well_audit = pd.read_csv(WELL_AUDIT_PATH)
well_audit["fold"] = pd.to_numeric(well_audit["fold"], errors="coerce").astype(int)
model_frame = residuals.merge(well_audit[["fold", "wellname"] + CONTEXT_FEATURES], on=["fold", "wellname"], how="left")

print(f"rows={len(model_frame):,}")
print(f"wells={model_frame['wellname'].nunique():,}")
print(model_frame.groupby("fold")["wellname"].nunique().rename("wells_by_fold").to_string())

rows=5,400
wells=773
fold
1    258
2    258
3    257


## 3. Candidate Contextual Hierarchy Arms

In [3]:
"Define regularized contextual residual-interaction arms."

CANDIDATES = [
    {"name": "baseline_no_context", "kind": "baseline", "alpha": 0.0, "shrink": 0.0, "prior": "zero contextual correction"},
    {"name": "context_main_ridge_a300_s025", "kind": "context_main", "alpha": 300.0, "shrink": 0.25, "prior": "descriptor main effects with strong ridge and 75 percent shrinkage"},
    {"name": "context_row_interactions_ridge_a500_s025", "kind": "context_row_interactions", "alpha": 500.0, "shrink": 0.25, "prior": "descriptor-by-row interactions with strong ridge and 75 percent shrinkage"},
    {"name": "context_row_interactions_ridge_a1500_s025", "kind": "context_row_interactions", "alpha": 1500.0, "shrink": 0.25, "prior": "descriptor-by-row interactions with very strong ridge and 75 percent shrinkage"},
    {"name": "context_row_interactions_ridge_a1500_s010", "kind": "context_row_interactions", "alpha": 1500.0, "shrink": 0.10, "prior": "descriptor-by-row interactions with very strong ridge and 90 percent shrinkage"},
    {"name": "context_poly2_ridge_a2000_s025", "kind": "context_poly2", "alpha": 2000.0, "shrink": 0.25, "prior": "quadratic contextual surface with very strong ridge"},
    {"name": "context_row_huber_a005_s025", "kind": "context_row_huber", "alpha": 0.05, "shrink": 0.25, "prior": "robust contextual row interactions with Huber loss and zero shrinkage"},
]

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def clipped(values):
    return np.clip(np.asarray(values, dtype=float), -OFFSET_CLIP, OFFSET_CLIP)

def add_interactions(frame):
    out = frame.copy()
    for context_col in CONTEXT_FEATURES:
        for row_col in ROW_FEATURES:
            out[f"{context_col}__x__{row_col}"] = out[context_col] * out[row_col]
    return out

CONTEXT_MAIN_FEATURES = CONTEXT_FEATURES
INTERACTION_FEATURES = CONTEXT_FEATURES + ROW_FEATURES + [f"{c}__x__{r}" for c in CONTEXT_FEATURES for r in ROW_FEATURES]

print(pd.DataFrame(CANDIDATES)[["name", "kind", "alpha", "shrink", "prior"]])

                                        name                      kind  \
0                        baseline_no_context                  baseline   
1               context_main_ridge_a300_s025              context_main   
2   context_row_interactions_ridge_a500_s025  context_row_interactions   
3  context_row_interactions_ridge_a1500_s025  context_row_interactions   
4  context_row_interactions_ridge_a1500_s010  context_row_interactions   
5             context_poly2_ridge_a2000_s025             context_poly2   
6                context_row_huber_a005_s025         context_row_huber   

     alpha  shrink                                              prior  
0     0.00    0.00                         zero contextual correction  
1   300.00    0.25  descriptor main effects with strong ridge and ...  
2   500.00    0.25  descriptor-by-row interactions with strong rid...  
3  1500.00    0.25  descriptor-by-row interactions with very stron...  
4  1500.00    0.10  descriptor-by-row interacti

## 4. Out-Of-Fold Evaluation

In [4]:
"Train contextual correction arms on other folds and score the held-out fold."

predicted_rows = []
fold_metric_rows = []
for candidate in CANDIDATES:
    for fold in sorted(model_frame["fold"].dropna().unique()):
        train_rows = model_frame.loc[~model_frame["fold"].eq(fold)].copy()
        valid_rows = model_frame.loc[model_frame["fold"].eq(fold)].copy()
        raw_correction = np.zeros(len(valid_rows), dtype=float)
        if candidate["kind"] == "context_main":
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler()), ("model", Ridge(alpha=candidate["alpha"]))])
            model.fit(train_rows[CONTEXT_MAIN_FEATURES], train_rows["baseline_residual"])
            raw_correction = model.predict(valid_rows[CONTEXT_MAIN_FEATURES])
        elif candidate["kind"] in ["context_row_interactions", "context_row_huber"]:
            train_x = add_interactions(train_rows)
            valid_x = add_interactions(valid_rows)
            if candidate["kind"] == "context_row_interactions":
                model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(alpha=candidate["alpha"]))])
            else:
                model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", HuberRegressor(alpha=candidate["alpha"], epsilon=1.35, max_iter=500))])
            model.fit(train_x[INTERACTION_FEATURES], train_x["baseline_residual"])
            raw_correction = model.predict(valid_x[INTERACTION_FEATURES])
        elif candidate["kind"] == "context_poly2":
            train_x = add_interactions(train_rows)
            valid_x = add_interactions(valid_rows)
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("poly", PolynomialFeatures(degree=2, include_bias=False)), ("model", Ridge(alpha=candidate["alpha"]))])
            model.fit(train_x[CONTEXT_FEATURES + ROW_FEATURES], train_x["baseline_residual"])
            raw_correction = model.predict(valid_x[CONTEXT_FEATURES + ROW_FEATURES])
        correction = candidate["shrink"] * clipped(raw_correction)
        y_pred_context = valid_rows["baseline_pred"].to_numpy(dtype=float) + correction
        eval_frame = valid_rows[["fold", "wellname", "row_idx", "y_true", "baseline_pred", "baseline_residual", "eval_progress", "distance_log1p", "abs_projected_delta_10"] + CONTEXT_FEATURES].copy()
        eval_frame["candidate"] = candidate["name"]
        eval_frame["raw_context_correction"] = raw_correction
        eval_frame["context_correction"] = correction
        eval_frame["y_pred"] = y_pred_context
        eval_frame["residual_after_context"] = eval_frame["y_true"] - eval_frame["y_pred"]
        predicted_rows.append(eval_frame)
        fold_metric_rows.append(
            {
                "candidate": candidate["name"],
                "fold": int(fold),
                "kind": candidate["kind"],
                "alpha": candidate["alpha"],
                "shrink": candidate["shrink"],
                "prior": candidate["prior"],
                "n_rows": int(len(valid_rows)),
                "n_wells": int(valid_rows["wellname"].nunique()),
                "rmse": rmse(valid_rows["y_true"], y_pred_context),
                "mae": float(mean_absolute_error(valid_rows["y_true"], y_pred_context)),
                "mean_context_correction": float(np.mean(correction)),
                "context_correction_std": float(np.std(correction)),
                "mean_residual_after_context": float(np.mean(eval_frame["residual_after_context"])),
            }
        )

predictions = pd.concat(predicted_rows, ignore_index=True)
fold_metrics = pd.DataFrame(fold_metric_rows)
summary = (
    fold_metrics.groupby(["candidate", "kind", "alpha", "shrink", "prior"], as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mean_abs_fold_bias=("mean_residual_after_context", lambda s: float(np.mean(np.abs(s)))),
        mean_context_correction_abs=("mean_context_correction", lambda s: float(np.mean(np.abs(s)))),
        context_correction_std_mean=("context_correction_std", "mean"),
    )
)
sample_baseline_rmse = float(summary.loc[summary["candidate"].eq("baseline_no_context"), "rmse_mean"].iloc[0])
summary["rmse_delta_vs_sample_baseline"] = summary["rmse_mean"] - sample_baseline_rmse
summary["rmse_delta_vs_full_reference"] = summary["rmse_mean"] - LOCAL_REFERENCE_RMSE
summary = summary.sort_values("rmse_mean")
summary

,candidate,kind,alpha,shrink,prior,rmse_mean,rmse_std,mae_mean,mean_abs_fold_bias,mean_context_correction_abs,context_correction_std_mean,rmse_delta_vs_sample_baseline,rmse_delta_vs_full_reference
2,context_poly2_ridge_a2000_s025,context_poly2,2000.00,0.25,quadratic contextual surface with very strong ...,14.732182,0.475714,10.422177,0.283212,0.049806,0.366977,-0.006363,0.373582
0,baseline_no_context,baseline,0.00,0.00,zero contextual correction,14.738545,0.487053,10.413072,0.303161,0.000000,0.000000,0.000000,0.379945
4,context_row_interactions_ridge_a1500_s010,context_row_interactions,1500.00,0.10,descriptor-by-row interactions with very stron...,14.741099,0.490348,10.416018,0.299312,0.024887,0.082202,0.002554,0.382499
6,context_row_interactions_ridge_a500_s025,context_row_interactions,500.00,0.25,descriptor-by-row interactions with strong rid...,14.744694,0.495076,10.420678,0.286543,0.061067,0.231186,0.006149,0.386094
5,context_row_interactions_ridge_a1500_s025,context_row_interactions,1500.00,0.25,descriptor-by-row interactions with very stron...,14.745877,0.495300,10.420805,0.293540,0.062218,0.205505,0.007332,0.387277
1,context_main_ridge_a300_s025,context_main,300.00,0.25,descriptor main effects with strong ridge and ...,14.748060,0.494260,10.421365,0.294703,0.055203,0.221877,0.009515,0.389460
3,context_row_huber_a005_s025,context_row_huber,0.05,0.25,robust contextual row interactions with Huber ...,14.752066,0.498037,10.430316,0.344057,0.049385,0.440367,0.013521,0.393466


## 5. Diagnostics And Decision

In [5]:
"Summarize contextual hierarchy diagnostics for the best arm."

best_candidate = summary.iloc[0]["candidate"]
best_row = summary.iloc[0]
baseline_row = summary.loc[summary["candidate"].eq("baseline_no_context")].iloc[0]
best_predictions = predictions.loc[predictions["candidate"].eq(best_candidate)].copy()
well_after = (
    best_predictions.groupby(["fold", "wellname"], as_index=False)
    .agg(
        n_rows=("residual_after_context", "size"),
        mean_residual_after_context=("residual_after_context", "mean"),
        mae_after_context=("residual_after_context", lambda s: float(np.mean(np.abs(s)))),
        rmse_after_context=("residual_after_context", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
        mean_context_correction=("context_correction", "mean"),
        context_correction_std=("context_correction", "std"),
    )
)
well_before = (
    model_frame.groupby(["fold", "wellname"], as_index=False)
    .agg(
        mean_residual_before=("baseline_residual", "mean"),
        rmse_before=("baseline_residual", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
    )
)
well_diagnostics = well_after.merge(well_before, on=["fold", "wellname"], how="left")
well_diagnostics["abs_mean_residual_before"] = well_diagnostics["mean_residual_before"].abs()
well_diagnostics["abs_mean_residual_after"] = well_diagnostics["mean_residual_after_context"].abs()
well_diagnostics["abs_bias_delta"] = well_diagnostics["abs_mean_residual_after"] - well_diagnostics["abs_mean_residual_before"]

findings = pd.DataFrame(
    [
        {
            "finding": "best_candidate",
            "value": f"{best_candidate} RMSE {best_row['rmse_mean']:.4f} vs sample baseline {baseline_row['rmse_mean']:.4f}",
            "modeling_implication": "Promote only if contextual interactions clear the hierarchy gate.",
        },
        {
            "finding": "sample_baseline_delta",
            "value": f"best candidate delta {best_row['rmse_delta_vs_sample_baseline']:.4f}",
            "modeling_implication": "Negative means descriptor-by-row context improved the same residual frame.",
        },
        {
            "finding": "well_bias_change",
            "value": f"median absolute well-bias delta {well_diagnostics['abs_bias_delta'].median():.4f}",
            "modeling_implication": "Context should not reduce row RMSE by moving bias between wells.",
        },
    ]
)

print(f"best_candidate={best_candidate}")
print(summary.to_string(index=False))

best_candidate=context_poly2_ridge_a2000_s025
                                candidate                     kind   alpha  shrink                                                                          prior  rmse_mean  rmse_std  mae_mean  mean_abs_fold_bias  mean_context_correction_abs  context_correction_std_mean  rmse_delta_vs_sample_baseline  rmse_delta_vs_full_reference
           context_poly2_ridge_a2000_s025            context_poly2 2000.00    0.25                            quadratic contextual surface with very strong ridge  14.732182  0.475714 10.422177            0.283212                     0.049806                     0.366977                      -0.006363                      0.373582
                      baseline_no_context                 baseline    0.00    0.00                                                     zero contextual correction  14.738545  0.487053 10.413072            0.303161                     0.000000                     0.000000                    

## 6. Export Artifacts

In [6]:
"Export contextual hierarchy metrics, diagnostics, report, and run bundle."

fold_metrics_path = ARTIFACT_TABLES / f"model__contextual_hierarchical_huber_fold_metrics__{DATE_STAMP}.csv"
summary_path = ARTIFACT_TABLES / f"model__contextual_hierarchical_huber_summary__{DATE_STAMP}.csv"
residual_sample_path = ARTIFACT_TABLES / f"model__contextual_hierarchical_huber_residual_sample__{DATE_STAMP}.csv"
well_diagnostics_path = ARTIFACT_TABLES / f"model__contextual_hierarchical_huber_well_diagnostics__{DATE_STAMP}.csv"
findings_path = ARTIFACT_TABLES / f"model__contextual_hierarchical_huber_findings__{DATE_STAMP}.csv"
comparison_path = ARTIFACT_TABLES / f"model__contextual_hierarchical_huber_comparison__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__contextual_hierarchical_huber_screen__{DATE_STAMP}.md"

fold_metrics.to_csv(fold_metrics_path, index=False)
summary.to_csv(summary_path, index=False)
best_predictions.to_csv(residual_sample_path, index=False)
well_diagnostics.to_csv(well_diagnostics_path, index=False)
findings.to_csv(findings_path, index=False)
comparison = pd.DataFrame(
    [
        {
            "date": DATE_STAMP,
            "model": best_candidate,
            "local_rmse": float(best_row["rmse_mean"]),
            "local_rmse_std": float(best_row["rmse_std"]),
            "delta_vs_sample_baseline": float(best_row["rmse_delta_vs_sample_baseline"]),
            "delta_vs_full_huber_hilbert_reference": float(best_row["rmse_delta_vs_full_reference"]),
            "public_lb_reference": PUBLIC_REFERENCE_RMSE,
            "decision": "promote_to_next_hierarchy" if best_row["rmse_delta_vs_sample_baseline"] <= -0.05 else "do_not_promote_yet",
            "notes": "Contextual descriptor-by-row residual correction on sampled out-of-fold residuals; no Kaggle submission.",
        }
    ]
)
comparison.to_csv(comparison_path, index=False)

def markdown_table(frame):
    display_frame = frame.copy()
    for col in display_frame.select_dtypes(include=["float"]).columns:
        display_frame[col] = display_frame[col].map(lambda value: "" if pd.isna(value) else f"{value:.4f}")
    columns = list(display_frame.columns)
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in display_frame.iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in columns) + " |")
    return "\n".join([header, divider] + rows)

report_lines = [
    "# Contextual Hierarchical Huber Screen",
    "",
    f"Date: {DATE_STAMP}",
    "",
    "## Purpose",
    "",
    "Test whether test-safe well descriptors can modulate row-level residual shape through regularized contextual interactions.",
    "",
    "## Priors And Assumptions",
    "",
    f"- Corrections are contextual residual offsets centered on zero and clipped to +/- {OFFSET_CLIP:.1f} TVT units before shrinkage.",
    "- Descriptor-by-row interactions approximate contextual hierarchy without raw well-ID effects.",
    "- Ridge/Huber penalties and fixed shrinkage are the empirical-Bayes prior toward zero correction.",
    "- Every arm trains on other folds only and scores the held-out fold.",
    "",
    "## Results",
    "",
    markdown_table(summary),
    "",
    "## Findings",
    "",
    markdown_table(findings),
    "",
    "## Decision",
    "",
    f"Best candidate `{best_candidate}` scored **{best_row['rmse_mean']:.4f}** RMSE, delta **{best_row['rmse_delta_vs_sample_baseline']:.4f}** vs the sample no-context baseline and **{best_row['rmse_delta_vs_full_reference']:.4f}** vs the stored full Huber + Hilbert reference.",
    "",
    "## Artifacts",
    "",
    f"- Fold metrics: `{fold_metrics_path}`",
    f"- Summary: `{summary_path}`",
    f"- Residual sample: `{residual_sample_path}`",
    f"- Well diagnostics: `{well_diagnostics_path}`",
    f"- Findings: `{findings_path}`",
    f"- Comparison row: `{comparison_path}`",
]
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

for path in [fold_metrics_path, summary_path, residual_sample_path, well_diagnostics_path, findings_path, comparison_path, report_path]:
    shutil.copy2(path, ARTIFACT_LOGS / path.name)
metadata = {
    "run_id": RUN_ID,
    "best_candidate": str(best_candidate),
    "best_rmse": float(best_row["rmse_mean"]),
    "best_delta_vs_sample_baseline": float(best_row["rmse_delta_vs_sample_baseline"]),
    "best_delta_vs_full_reference": float(best_row["rmse_delta_vs_full_reference"]),
    "reference_rmse": LOCAL_REFERENCE_RMSE,
    "public_reference_rmse": PUBLIC_REFERENCE_RMSE,
    "decision": str(comparison.iloc[0]["decision"]),
}
(ARTIFACT_LOGS / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(report_path)
print(summary_path)
print(fold_metrics_path)

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\model__contextual_hierarchical_huber_screen__2026-06-23.md
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__contextual_hierarchical_huber_summary__2026-06-23.csv
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__contextual_hierarchical_huber_fold_metrics__2026-06-23.csv


## Results

This notebook screens contextual descriptor-by-row hierarchy. It does not create a Kaggle submission unless the same-sample residual correction clears the promotion gate.